# Data Mining and Preparation using Web Scraping
Handling sensitive data, ethically. 

•	https://worldpopulationreview.com/state-rankings/waiting-period-for-guns-by-state#title

## Code Attribution
<ul>
  <li> Title: Python for Data Analysis 
  <li> Author: Wes McKinney
  <li> Date: 02 November 2025
  <li> Modified By: Charlotte Alley
  <li> Description: Data Loading, Storage, and File Formats

<ul>
  <li> Title: The Data Wrangling Workshop
  <li> Author: Brian Lipp, Shubhadeep Roychowdhury, Dr. Tirthajyoti Sarkar
  <li> Date: 02 November 2025
  <li> Modified By: Charlotte Alley
  <li> Description: Data Sources & Data Wrangling

In [1]:
from io import StringIO
import pandas as pd
from bs4 import BeautifulSoup

# Closes file right after reading
with open('Waiting Period for Guns by State 2025.html', 'r', 
          encoding='utf-8') as fd:
    soup = BeautifulSoup(fd)

In [2]:
# Finds the table by class name
table = soup.find("table", class_="wpr-table")

# Reads the table HTML into a DataFrame, wraps in StringIO to avoid warning
waiting_df = pd.read_html((StringIO(str(table))))[0]

# Outputs first five rows
waiting_df.head()

,Unnamed: 0,State,Gun Purchase Waiting Periods↓,Waiting Period Info
0,NaN,Minnesota,30,Wait applies to handguns and assault weapons
1,NaN,Hawaii,14,Wait applies to all firearms
2,NaN,California,10,Wait applies to all firearms
3,NaN,District of Columbia,10,Wait applies to all firearms
4,NaN,Washington,10,Wait applies to all firearms. Duration is 10 d...


## Step 1: Remove Unnecessary Columns
The first column in the dataframe appears to be entirely NaN values, and judging from the website itself, it's meant to represent the state flag images. This information is not only incomplete, but it's unnecessary for our analysis, so I will be removing the column, entirely. 

In [3]:
waiting_df = waiting_df.drop(['Unnamed: 0'], axis=1)
waiting_df.head()

,State,Gun Purchase Waiting Periods↓,Waiting Period Info
0,Minnesota,30,Wait applies to handguns and assault weapons
1,Hawaii,14,Wait applies to all firearms
2,California,10,Wait applies to all firearms
3,District of Columbia,10,Wait applies to all firearms
4,Washington,10,Wait applies to all firearms. Duration is 10 d...


In [4]:
# Takes a quick look to ensure all states are included and spelled correctly
waiting_df['State'].unique()

array(['Minnesota', 'Hawaii', 'California', 'District of Columbia',
       'Washington', 'Maryland', 'New Jersey', 'New Mexico',
       'Rhode Island', 'Colorado', 'Florida', 'Illinois', 'Maine',
       'Vermont', 'Alabama', 'Alaska', 'Arizona', 'Arkansas',
       'Connecticut', 'Delaware', 'Georgia', 'Idaho', 'Indiana', 'Iowa',
       'Kansas', 'Kentucky', 'Louisiana', 'Massachusetts', 'Michigan',
       'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada',
       'New Hampshire', 'New York', 'North Carolina', 'North Dakota',
       'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'South Carolina',
       'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Virginia',
       'West Virginia', 'Wisconsin', 'Wyoming'], dtype=object)

In [5]:
# Checks for null values and dtypes
waiting_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 3 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   State                          51 non-null     object
 1   Gun Purchase Waiting Periods↓  51 non-null     int64 
 2   Waiting Period Info            14 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.3+ KB


In [6]:
# Checks descriptive statistics, scale and presence of outliers  
waiting_df.describe()

,Gun Purchase Waiting Periods↓
count,51.000000
mean,2.294118
std,5.231803
min,0.000000
25%,0.000000
50%,0.000000
75%,3.000000
max,30.000000


In [7]:
# Checks for duplicate rows in dataset 
waiting_df.duplicated().sum()

np.int64(0)

## Step 2: Filling Missing Values
Since this dataframe only has 51 rows, I was able to print the entire dataframe to confirm that the missing values only existed for states that have no waiting period. With that information, I filled the missing values with 'No Waiting Period'. 

In [8]:
# Outputs first 51 rows, which is the entire dataframe in this case
waiting_df.head(51)

,State,Gun Purchase Waiting Periods↓,Waiting Period Info
0,Minnesota,30,Wait applies to handguns and assault weapons
1,Hawaii,14,Wait applies to all firearms
2,California,10,Wait applies to all firearms
3,District of Columbia,10,Wait applies to all firearms
4,Washington,10,Wait applies to all firearms. Duration is 10 d...
5,Maryland,7,Wait applies to handguns only
6,New Jersey,7,Wait applies to handguns only
7,New Mexico,7,Wait applies to all firearms
8,Rhode Island,7,Wait applies to all firearms. Wait is waived f...
9,Colorado,3,Wait applies to all firearms


In [9]:
# Replaces NaN's with 'No Waiting Period'
waiting_df = waiting_df.fillna('No Waiting Period')

# Confirms that there are no longer any missing values in the 'Waiting Period
# Info' column
waiting_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 3 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   State                          51 non-null     object
 1   Gun Purchase Waiting Periods↓  51 non-null     int64 
 2   Waiting Period Info            51 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.3+ KB


## Step 3: Recode Waiting Period Info
Since the goal of this project is data visualization, the current state of the 'Waiting Period Info' column is too lengthy for effective visualization. I'm going to create a map to shorten the information in this column, so that it will fit on a legend when visualized. There is some context loss that will occur with this transformation, so communicating this transformation and any relevent context with the results will be crucial.

In [10]:
# Prints values and their counts, to be used for future mapping
waiting_df['Waiting Period Info'].value_counts()

Waiting Period Info
No Waiting Period                                                                                                                                                37
Wait applies to all firearms                                                                                                                                      8
Wait applies to handguns only                                                                                                                                     2
Wait applies to all firearms. Duration is 10 days for semiautomatic rifles, the lesser of 3 days or until background check completes for other gun types          1
Wait applies to handguns and assault weapons                                                                                                                      1
Wait applies to all firearms. Wait is waived for law enforcement officials. After seven days, sale must be completed unless background check disqualifies it.   

In [11]:
# Creates dictionary with replacement values for column
mapping = {
    'No Waiting Period': 'No Waiting Period',
    'Wait applies to all firearms': 'All Firearms',
    'Wait applies to handguns only': 'Handguns Only',
    ('Wait applies to all firearms. Duration is 10 days for semiautomatic '
     'rifles, the lesser of 3 days or until background check completes for '
     'other gun types'): 'All Firearms',
    'Wait applies to handguns and assault weapons': 
    'Handguns/Assault Weapons Only',
    ('Wait applies to all firearms. Wait is waived for law enforcement '
     'officials. After seven days, sale must be completed unless background '
     'check disqualifies it.'): 'All Firearms',
    ('Wait applies to all firearms. Duration is three days or the time it '
     'takes to complete required background checks, whichever is longer'): 
    'All Firearms'
}

# Uses map to replace values in column
waiting_df['Gun Type'] = waiting_df['Waiting Period Info'].map(mapping)

# Drops 'Waiting Period Info'
waiting_df = waiting_df.drop('Waiting Period Info', axis=1)

# Outputs entire data frame to ensure replacements were successful
waiting_df.head(51)

,State,Gun Purchase Waiting Periods↓,Gun Type
0,Minnesota,30,Handguns/Assault Weapons Only
1,Hawaii,14,All Firearms
2,California,10,All Firearms
3,District of Columbia,10,All Firearms
4,Washington,10,All Firearms
5,Maryland,7,Handguns Only
6,New Jersey,7,Handguns Only
7,New Mexico,7,All Firearms
8,Rhode Island,7,All Firearms
9,Colorado,3,All Firearms


## Step 4: Create Boolean Column for 'Has Waiting Period'
For visualization purposes, it makes sense to have a column that clearly indicates whether a state has a gun purchasing waiting period or not. Creating the column itself is very easy, since we know that any waiting period above 0 days would signify they have a waiting period.

In [12]:
waiting_df['Has Waiting Period'] = waiting_df[
    'Gun Purchase Waiting Periods↓']> 0
waiting_df.head(10)

,State,Gun Purchase Waiting Periods↓,Gun Type,Has Waiting Period
0,Minnesota,30,Handguns/Assault Weapons Only,True
1,Hawaii,14,All Firearms,True
2,California,10,All Firearms,True
3,District of Columbia,10,All Firearms,True
4,Washington,10,All Firearms,True
5,Maryland,7,Handguns Only,True
6,New Jersey,7,Handguns Only,True
7,New Mexico,7,All Firearms,True
8,Rhode Island,7,All Firearms,True
9,Colorado,3,All Firearms,True


## Step 5: Reorder and Rename Columns
I think that the 'Has Waiting Period' column belongs earlier in the dataframe, to improve readability with the story that the data is telling. I'm also going to rename the 'Gun Purchase Waiting Periods' column to something more intuitive for the values it holds. 

In [13]:
# Reorders columns
waiting_df = waiting_df[['State', 'Has Waiting Period', 
                         'Gun Purchase Waiting Periods↓', 'Gun Type']]

# Outputs first five rows
waiting_df.head()

,State,Has Waiting Period,Gun Purchase Waiting Periods↓,Gun Type
0,Minnesota,True,30,Handguns/Assault Weapons Only
1,Hawaii,True,14,All Firearms
2,California,True,10,All Firearms
3,District of Columbia,True,10,All Firearms
4,Washington,True,10,All Firearms


In [14]:
# Renames columns 
waiting_df.columns = ['State', 'Has Waiting Period', 'Wait in Days', 
                      'Gun Type']

# Human readable dataset
waiting_df.head()

# Save to csv file for database
waiting_df.to_csv('gun_waiting_periods.csv', index=False)

## Ethical Considerations
The changes made to the data include: removing unnecessary columns, filling missing values, recoding the 'Waiting Period Info' variable, creating a boolean column for 'Has Waiting Period', and reordering and renaming the columns. Looking at the .describe() table for the dataset, there are clearly outliers, with a 'Gun Purchase Waiting Period' mean of 2.29, and a maximum value of 30. I chose not to remove them, because they are meaningful in the context of this dataset.

There aren't any explicit legal or regulatory restrictions mentioned on the source's website. The waiting period laws are public knowledge. I did notice that the waiting period in MN has changed since the site was initialized. It was updated on the source, but I do think it's worth noting that any updates need to continue to be represented correctly on the dataset and in the footnotes section.

There are many risks that could be created with the transformations completed. Removing unnecessary columns must be done with thought and care, otherwise you risk introducing bias into the dataset. In this case, the column we removed without replacement was a duplication of the index column, and can be safely removed without creating any information lost. Whenever you're filling in missing values, you're creating an opportunity to introduce inaccurate assumptions if the absense of the data is meaningful. In this example, the absense of the data was meaningful, because it signified that there was no waiting period in place. Recoding 'Waiting Period Info' risks oversimplifying the detailed and nuanced legal differences between waiting periods. It's important for the context of these transformations to be shared with any conclusions, to deliver a completed picture to the audience. Creating the boolean column for the presence of a waiting period could have a similar effect, omitting nuance that would otherwise be present with more lengthy explanations. Reordering and renaming columns has the potential to create confusion if not done in a way that is easily understandable by the audience. These transformations could lead to misunderstandings or misinterpretations if not included in any conclusions, or users do not refer back to original, detailed data.

In cleaning and transforming the data, I made the assumption that the values on the dataset were accurate, and relevant to the problem that we are exploring. The data is sourced from worldpopulationreview.com, which collects official state firearm regulations and waiting period info. The site cites Gifford Law Center, Everytown Research and Fortune as sources for the data compilation. All sources are still accessible and available for public use. The dataset doesn't contain personal or private information, avoiding any privacy or ethical concerns, and includes footnotes with added context, increasing reliability.

Moving forward with the ethical implications that we have identified, clearly documenting assumptions and transformations for transparency, avoiding interpretations without considering the nuanced complexities of gun laws, and acknowledging the limitations in representing complex legal laws as boolean features will prevent bias and increase understanding. 